# 2D Allen–Cahn — Bimodal Posterior

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/10_allen_cahn_bimodal.ipynb)

This notebook reproduces **Example 4, Figures 5–8** from Alberts & Bilionis (2023).

We consider the 2D Allen–Cahn energy functional on $[-1, 1]^2$:

$$U_\varepsilon[\phi] = \int_{[-1,1]^2} \left[
    \frac{\varepsilon^2}{2}|\nabla\phi|^2 + W(\phi) - f\phi
\right] \mathrm{d}x$$

where $W(\phi) = \frac{1}{4}(1-\phi^2)^2$ is the double-well potential and
$\varepsilon = 0.01$ is the interface width.  The double-well drives the posterior
toward **two symmetric modes** ($\phi \approx +1$ or $\phi \approx -1$ over most of the
domain).

We observe $\phi$ on 3 of the 4 boundaries (left, right, bottom) but leave the top
boundary unobserved, creating posterior uncertainty there.

**Sampler:** NumPyro NUTS rather than SGLD, because HMC handles multimodal targets more
reliably.  A Gaussian mixture model (GMM) separates the two posterior modes.

**Estimated runtime:** ~3 minutes on CPU (500 warmup + 1000 NUTS samples, 9 parameters).

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git
%pip install -q numpyro

import time
import jax
jax.config.update('jax_enable_x64', True)
import numpy as np
import matplotlib.pyplot as plt
import numpyro
print('NumPyro:', numpyro.__version__)

from pipelines.phase_d_allen_cahn import run_phase_d_allen_cahn

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'epsilon':              0.01,   # interface width epsilon                [1e-3, 0.2]
    'beta':                 100.0,  # physics trust                          [1.0, 1e4]
    'max_freq':             1,      # 2D Fourier freq -> 9 params            [1, 4]
    'n_obs_per_boundary':   15,     # points per observed boundary           [3, 100]
    'noise_std':            0.01,   #                                        [1e-4, 0.5]
    'n_quad_per_dim':       40,     # 40x40 = 1600 quadrature points         [10, 80]
    'num_warmup':           500,    # NUTS warmup                            [100, 5000]
    'num_samples':          1000,   # production samples                     [100, 10000]
    'num_chains':           1,      #                                        [1, 8]
    'target_accept_prob':   0.8,    #                                        [0.6, 0.99]
    'max_tree_depth':       10,     #                                        [6, 14]
    'prior_std':            10.0,   #                                        [0.1, 100]
    'n_gmm_components':     2,      # GMM modes                              [1, 5]
    'n_grid_per_dim':       50,     #                                        [20, 200]
}

## Run NUTS Sampler

In [ ]:
t_start = time.perf_counter()

d_result = run_phase_d_allen_cahn(
    cfg=CONFIG,
    device_preference=jax.default_backend(),
    save_outputs=False,
)

elapsed = time.perf_counter() - t_start
print(f'Status:  {d_result["status"]}')
print(f'Runtime: {elapsed:.1f} s  ({elapsed/60:.1f} min)')

mode_indices = d_result.get('mode_indices', {})
mode_labels  = d_result.get('mode_labels', np.array([]))
theta_samples = d_result.get('theta_samples', np.zeros((0, 9)))

print(f'Total samples: {len(theta_samples)}')
print('Mode split:')
for k, idx in mode_indices.items():
    frac = len(idx) / max(1, len(theta_samples))
    print(f'  Mode {k}: {len(idx)} samples ({frac*100:.1f}%)')

## Results

In [ ]:
# Extract result arrays
gx, gy       = d_result['grid_coords']
phi_truth    = d_result['phi_truth']    # (n_grid, n_grid)
phi_median   = d_result['phi_median']   # (n_grid, n_grid)
phi_mean_all = d_result.get('phi_mean', phi_median)  # (n_grid, n_grid)
mode_fields  = d_result['mode_fields']  # dict: int -> (n_grid, n_grid)

n_modes = len(mode_fields)
gxx, gyy = np.meshgrid(gx, gy)

# ----------------------------------------------------------------
# Figure 7: ground truth + posterior median + per-mode means
# ----------------------------------------------------------------
ncols = 2 + n_modes
fig7, axes7 = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4.5))

vmin_all = min(phi_truth.min(), phi_median.min(),
               min(v.min() for v in mode_fields.values()))
vmax_all = max(phi_truth.max(), phi_median.max(),
               max(v.max() for v in mode_fields.values()))
# Symmetric colour scale for Allen-Cahn
vlim = max(abs(vmin_all), abs(vmax_all))
kw = dict(cmap='RdBu_r', vmin=-vlim, vmax=vlim, shading='auto')

axes7[0].pcolormesh(gxx, gyy, phi_truth, **kw)
axes7[0].set_title('Ground Truth', fontsize=11)
axes7[0].set_aspect('equal')

im = axes7[1].pcolormesh(gxx, gyy, phi_median, **kw)
axes7[1].set_title('Posterior Median (all)', fontsize=11)
axes7[1].set_aspect('equal')

for col_i, (k, mf) in enumerate(sorted(mode_fields.items())):
    n_in_mode = len(mode_indices.get(k, []))
    ax = axes7[2 + col_i]
    ax.pcolormesh(gxx, gyy, mf, **kw)
    ax.set_title(f'Mode {k} mean\n({n_in_mode} samples)', fontsize=11)
    ax.set_aspect('equal')

fig7.colorbar(im, ax=axes7.tolist(), shrink=0.75, label='$\\phi$')
fig7.suptitle('Allen\u2013Cahn: Ground Truth and Per-Mode Predictions (Fig. 7)', fontsize=12)
fig7.tight_layout()
show_fig(fig7)

# ----------------------------------------------------------------
# Figure 8: absolute errors
# ----------------------------------------------------------------
n_err_panels = 2 + n_modes  # |mean-truth|, |median-truth|, |mode_k-truth|
fig8, axes8 = plt.subplots(1, n_err_panels, figsize=(4.5 * n_err_panels, 4.5))

err_kw = dict(cmap='Oranges', vmin=0, shading='auto')

im8_0 = axes8[0].pcolormesh(gxx, gyy, np.abs(phi_mean_all - phi_truth), **err_kw)
axes8[0].set_title('|Mean − Truth|', fontsize=11)
axes8[0].set_aspect('equal')
fig8.colorbar(im8_0, ax=axes8[0], shrink=0.9)

im8_1 = axes8[1].pcolormesh(gxx, gyy, np.abs(phi_median - phi_truth), **err_kw)
axes8[1].set_title('|Median − Truth|', fontsize=11)
axes8[1].set_aspect('equal')
fig8.colorbar(im8_1, ax=axes8[1], shrink=0.9)

for col_i, (k, mf) in enumerate(sorted(mode_fields.items())):
    ax = axes8[2 + col_i]
    im8_k = ax.pcolormesh(gxx, gyy, np.abs(mf - phi_truth), **err_kw)
    ax.set_title(f'|Mode {k} mean − Truth|', fontsize=11)
    ax.set_aspect('equal')
    fig8.colorbar(im8_k, ax=ax, shrink=0.9)

fig8.suptitle('Allen\u2013Cahn: Absolute Errors (Fig. 8)', fontsize=12)
fig8.tight_layout()
show_fig(fig8)

In [ ]:
# ----------------------------------------------------------------
# Figure 6: 3x3 bimodal marginal histograms (9 Fourier parameters)
# ----------------------------------------------------------------
theta_np = np.asarray(theta_samples)
n_params  = theta_np.shape[1]
n_show    = min(n_params, 9)
nrows, ncols_hist = 3, 3

# Colour palette: one colour per GMM mode
mode_colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']

fig6, axes6 = plt.subplots(nrows, ncols_hist, figsize=(4 * ncols_hist, 3.5 * nrows))
axes6_flat = axes6.ravel()

for pi in range(n_show):
    ax = axes6_flat[pi]
    # All samples (grey background)
    ax.hist(theta_np[:, pi], bins=40, color='lightgrey', edgecolor='white',
            density=True, alpha=0.6, label='All')
    # Per-mode overlays
    for k, idx in sorted(mode_indices.items()):
        if len(idx) == 0:
            continue
        color = mode_colors[k % len(mode_colors)]
        ax.hist(theta_np[idx, pi], bins=30, color=color, edgecolor='white',
                density=True, alpha=0.65, label=f'Mode {k}')
    ax.set_xlabel(f'$\\theta_{{{pi+1}}}$', fontsize=10)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(f'Param {pi+1}', fontsize=10)
    if pi == 0:
        ax.legend(fontsize=8, loc='upper right')

# Hide unused panels
for pi in range(n_show, len(axes6_flat)):
    axes6_flat[pi].set_visible(False)

fig6.suptitle('Allen\u2013Cahn: Bimodal Marginal Posteriors (Fig. 6)\n'
              '(per-mode coloured; ~50/50 split expected)', fontsize=12)
fig6.tight_layout()
show_fig(fig6)

## Interpretation

**Bimodal structure:** The Allen–Cahn energy has two symmetric global minima
($\phi \approx +1$ and $\phi \approx -1$) connected by interface layers of width
$\sim \varepsilon$.  With $\varepsilon = 0.01$ the two modes are nearly disconnected in
parameter space, making standard MCMC prone to mode trapping.  NUTS with `num_chains=1`
may sample predominantly from one mode; increase `num_chains` or run multiple independent
chains to ensure both modes are represented.

**Expected mode split:** approximately 50/50 for the symmetric problem with the default
CONFIG.  The exact split depends on the random seed and which mode the warmup phase
converges to.

**Per-mode predictions (Fig. 7):** each mode gives a distinct field pattern.  The
combined median (all samples) blurs the two modes and generally does not correspond to a
physically meaningful solution; the per-mode means are more informative.

**Absolute errors (Fig. 8):** the per-mode error is typically much lower than the
all-sample median error, confirming that mode separation is essential for accurate
prediction.